<a href="https://colab.research.google.com/github/25bca089-Razi/BirdVision-AI/blob/main/AI_ML_prj_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ============================================================
# BIRDVISION AI - COMPLETE BIRD RECOGNITION PROJECT
# Google Colab | Python | Gradio | Hugging Face CLIP
# ============================================================

!pip install -q gradio transformers torch pillow pandas

import gradio as gr
import torch
import pandas as pd
import tempfile
import html
from datetime import datetime
from transformers import pipeline


# ============================================================
# 1. BIRD CATEGORIES
# ============================================================

BIRDS = [
    "sparrow", "eagle", "parrot", "pigeon", "crow",
    "owl", "peacock", "penguin", "flamingo", "kingfisher",
    "woodpecker", "hummingbird", "ostrich", "swan", "duck",
    "hen", "rooster", "turkey", "goose", "vulture",
    "hawk", "falcon", "crane", "heron", "stork",
    "seagull", "pelican", "toucan", "macaw", "cockatoo",
    "canary", "finch", "nightingale", "robin", "blue jay",
    "cardinal", "magpie", "parakeet", "albatross", "kiwi"
]

LABELS = [f"a photo of a {bird} bird" for bird in BIRDS]


# ============================================================
# 2. BIRD FACTS
# ============================================================

FACTS = {
    "sparrow": "Sparrows are small birds commonly found near human settlements.",
    "eagle": "Eagles are powerful birds of prey with excellent eyesight.",
    "parrot": "Parrots are intelligent birds known for colorful feathers.",
    "pigeon": "Pigeons are adaptable birds with excellent navigation abilities.",
    "crow": "Crows are highly intelligent birds capable of solving problems.",
    "owl": "Owls are nocturnal birds with excellent hearing.",
    "peacock": "Peacocks are famous for their colorful tail feathers.",
    "penguin": "Penguins are flightless birds and excellent swimmers.",
    "flamingo": "Flamingos are known for their pink feathers and long legs.",
    "kingfisher": "Kingfishers are colorful birds that catch fish.",
    "woodpecker": "Woodpeckers use their strong beaks to search for insects.",
    "hummingbird": "Hummingbirds can hover while feeding on nectar.",
    "ostrich": "Ostriches are the largest living birds.",
    "swan": "Swans are large waterbirds known for their graceful appearance.",
    "duck": "Ducks are waterbirds with webbed feet.",
    "hen": "Hens are domesticated female chickens that lay eggs.",
    "rooster": "Roosters are male chickens known for crowing.",
    "turkey": "Turkeys are large birds native to North America.",
    "goose": "Geese are social waterbirds that travel in groups.",
    "vulture": "Vultures are scavenging birds that help clean the environment.",
    "hawk": "Hawks are birds of prey with sharp talons.",
    "falcon": "Falcons are fast-flying birds of prey.",
    "crane": "Cranes are tall birds with long legs.",
    "heron": "Herons are long-legged birds found near wetlands.",
    "stork": "Storks are large wading birds with long bills.",
    "seagull": "Seagulls are commonly found near oceans.",
    "pelican": "Pelicans have large bills with expandable throat pouches.",
    "toucan": "Toucans are tropical birds with colorful bills.",
    "macaw": "Macaws are large parrots with bright feathers.",
    "cockatoo": "Cockatoos are intelligent parrots with distinctive crests.",
    "canary": "Canaries are small songbirds known for beautiful singing.",
    "finch": "Finches are small birds with strong seed-eating beaks.",
    "nightingale": "Nightingales are famous for their beautiful songs.",
    "robin": "Robins are songbirds often recognized by their reddish breast.",
    "blue jay": "Blue jays are intelligent birds with blue feathers.",
    "cardinal": "Cardinals are songbirds known for bright red plumage.",
    "magpie": "Magpies are intelligent black and white birds.",
    "parakeet": "Parakeets are colorful small to medium-sized parrots.",
    "albatross": "Albatrosses are seabirds with impressive wingspans.",
    "kiwi": "Kiwis are flightless birds native to New Zealand."
}


# ============================================================
# 3. LOAD AI MODEL
# ============================================================

print("Loading BirdVision AI model...")

device = 0 if torch.cuda.is_available() else -1

print("Device:", "GPU" if device == 0 else "CPU")

classifier = pipeline(
    "zero-shot-image-classification",
    model="openai/clip-vit-base-patch32",
    device=device
)

print("AI MODEL LOADED SUCCESSFULLY!")


# ============================================================
# 4. DASHBOARD DESIGN
# ============================================================

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');

.gradio-container {
    font-family: Inter, sans-serif !important;
    background: #f0fdf4 !important;
    max-width: 1250px !important;
}

#header {
    background: linear-gradient(120deg,#064e3b,#059669,#0f766e);
    padding: 30px;
    border-radius: 18px;
    margin-bottom: 20px;
    color: white !important;
}

#header h1 {
    color: white !important;
    font-size: 36px;
    font-weight: 800;
}

#header p {
    color: #d1fae5 !important;
}

.stat {
    background: white;
    padding: 18px;
    border-radius: 14px;
    border-left: 4px solid #059669;
    box-shadow: 0 4px 12px #00000010;
}

.stat h2 {
    color: #047857 !important;
}

.stat p, .stat small {
    color: #475569 !important;
}

.result {
    background: white;
    padding: 24px;
    border-radius: 16px;
    border-top: 5px solid #059669;
    box-shadow: 0 5px 20px #00000012;
}

.result h1 {
    color: #047857 !important;
    text-align: center;
}

.result h3 {
    color: #065f46 !important;
}

.bird {
    font-size: 42px;
    text-align: center;
}

.score {
    color: #047857 !important;
    font-size: 38px;
    font-weight: 800;
    text-align: center;
}

.muted {
    color: #475569 !important;
    text-align: center;
}

.fact {
    background: #ecfdf5;
    color: #065f46 !important;
    padding: 15px;
    border-radius: 10px;
    line-height: 1.7;
}

.chart {
    background: white;
    padding: 20px;
    border-radius: 15px;
}

.chart h3 {
    color: #065f46 !important;
}

.bar-row {
    margin: 17px 0;
}

.bar-label {
    display: flex;
    justify-content: space-between;
    font-size: 13px;
    margin-bottom: 7px;
    color: #14532d;
}

.track {
    height: 12px;
    background: #d1fae5;
    border-radius: 20px;
    overflow: hidden;
}

.fill {
    height: 100%;
    background: linear-gradient(90deg,#059669,#34d399);
    border-radius: 20px;
    animation: grow 1s ease;
}

@keyframes grow {
    from {width:0;}
}

.history {
    background: white;
    padding: 20px;
    border-radius: 15px;
}

.history-row {
    display: flex;
    justify-content: space-between;
    gap: 10px;
    padding: 12px;
    border-bottom: 1px solid #d1fae5;
    font-size: 12px;
    color: #14532d;
}

#analyze {
    background: linear-gradient(100deg,#059669,#0f766e) !important;
    color: white !important;
    font-weight: 700 !important;
    border: none !important;
}

#analyze span {
    color: white !important;
}

footer {
    display: none !important;
}
"""


# ============================================================
# 5. BIRD RECOGNITION FUNCTION
# ============================================================

def recognize_bird(image, history):

    if image is None:
        return (
            "<div class='result'><h2>Please upload a bird image!</h2></div>",
            "",
            pd.DataFrame(columns=["Rank", "Bird", "Score"]),
            "<p>No recognition history available.</p>",
            history or [],
            None
        )

    image = image.convert("RGB")

    predictions = classifier(
        image,
        candidate_labels=LABELS
    )

    best = predictions[0]

    bird = best["label"].replace(
        "a photo of a ", ""
    ).replace(" bird", "").title()

    score = float(best["score"]) * 100

    facts = FACTS.get(
        bird.lower(),
        "Discover interesting information about this bird."
    )

    # RESULT CARD

    result_html = f"""
    <div class="result">
        <div class="bird">🐦</div>
        <p class="muted">IDENTIFIED BIRD SPECIES</p>
        <h1>{html.escape(bird.upper())}</h1>
        <div class="score">{score:.2f}%</div>
        <p class="muted">Similarity Score</p>
        <hr>
        <h3>🌿 Bird Information</h3>
        <div class="fact">{html.escape(facts)}</div>
        <br>
        <small class="muted">
        Similarity scores are not calibrated probabilities.
        </small>
    </div>
    """

    # TOP 5 PREDICTIONS

    chart = """
    <div class="chart">
    <h3>📊 Top 5 Bird Predictions</h3>
    """

    rows = []

    for i, item in enumerate(predictions[:5], 1):

        name = item["label"].replace(
            "a photo of a ", ""
        ).replace(" bird", "").title()

        percentage = float(item["score"]) * 100

        chart += f"""
        <div class="bar-row">
            <div class="bar-label">
                <span>{i}. {html.escape(name)}</span>
                <b>{percentage:.2f}%</b>
            </div>
            <div class="track">
                <div class="fill" style="width:{percentage:.2f}%"></div>
            </div>
        </div>
        """

        rows.append([
            i,
            name,
            f"{percentage:.2f}%"
        ])

    chart += "</div>"

    table = pd.DataFrame(
        rows,
        columns=["Rank", "Bird", "Score"]
    )

    # HISTORY

    new_history = list(history or [])

    new_history.append({
        "Date": datetime.now().strftime("%d-%m-%Y %H:%M"),
        "Bird": bird,
        "Score": f"{score:.2f}%"
    })

    history_html = """
    <div class="history">
    <h3>📚 Bird Recognition History</h3>
    """

    for item in reversed(new_history[-10:]):

        history_html += f"""
        <div class="history-row">
            <span>🐦 {html.escape(item['Bird'])}</span>
            <b>{html.escape(item['Score'])}</b>
            <small>{html.escape(item['Date'])}</small>
        </div>
        """

    history_html += "</div>"

    # DOWNLOAD REPORT

    report = f"""
BIRDVISION AI REPORT
================================

Date: {datetime.now()}

Identified Bird: {bird}
Similarity Score: {score:.2f}%

TOP 5 BIRD PREDICTIONS
================================
"""

    for row in rows:
        report += f"{row[0]}. {row[1]} - {row[2]}\n"

    report += f"""

BIRD INFORMATION
================================
{facts}
"""

    file = tempfile.NamedTemporaryFile(
        mode="w",
        suffix=".txt",
        delete=False,
        encoding="utf-8"
    )

    file.write(report)
    file.close()

    return (
        result_html,
        chart,
        table,
        history_html,
        new_history,
        file.name
    )


# ============================================================
# 6. RESET FUNCTION
# ============================================================

def reset_dashboard():

    return (
        None,
        """
        <div class='result'>
            <h2>🐦 Ready for Bird Recognition</h2>
            <p>Upload a bird image to begin.</p>
        </div>
        """,
        "",
        pd.DataFrame(columns=["Rank", "Bird", "Score"]),
        "<p>No recognition history available.</p>",
        [],
        None
    )


# ============================================================
# 7. BUILD GRADIO DASHBOARD
# ============================================================

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="emerald",
        secondary_hue="green"
    ),
    css=CSS,
    title="BirdVision AI"
) as demo:

    history_state = gr.State([])

    # HEADER

    gr.HTML("""
    <div id="header">
        <h1>🐦 BirdVision AI</h1>
        <p>Intelligent Bird Species Recognition Dashboard</p>
        <p>Powered by Hugging Face CLIP Vision Model</p>
        <span>🟢 AI SYSTEM ONLINE</span>
    </div>
    """)

    # STATISTICS

    with gr.Row():

        gr.HTML("""
        <div class="stat">
            <small>AI MODEL</small>
            <h2>CLIP</h2>
            <p>Vision Recognition</p>
        </div>
        """)

        gr.HTML("""
        <div class="stat">
            <small>BIRD CATEGORIES</small>
            <h2>40</h2>
            <p>Supported species labels</p>
        </div>
        """)

        gr.HTML("""
        <div class="stat">
            <small>PREDICTIONS</small>
            <h2>TOP 5</h2>
            <p>Ranked bird results</p>
        </div>
        """)

    # IMAGE INPUT

    gr.Markdown("## 🔍 Bird Species Recognition")

    with gr.Row():

        with gr.Column():

            image_input = gr.Image(
                label="Upload Bird Image",
                type="pil",
                height=350
            )

            analyze_btn = gr.Button(
                "🐦 IDENTIFY BIRD",
                variant="primary",
                elem_id="analyze"
            )

            reset_btn = gr.Button(
                "↻ RESET DASHBOARD"
            )

            gr.Markdown(
                "Supported formats: JPG, JPEG, PNG, WEBP"
            )

        with gr.Column():

            result_output = gr.HTML(
                """
                <div class="result">
                    <h2>🐦 Ready for Recognition</h2>
                    <p>Upload a bird image to begin.</p>
                </div>
                """
            )

    # ANALYTICS

    gr.Markdown("---")
    gr.Markdown("## 📊 Bird Prediction Analytics")

    chart_output = gr.HTML()

    prediction_table = gr.Dataframe(
        headers=["Rank", "Bird", "Score"],
        interactive=False,
        label="Top 5 Bird Predictions"
    )

    # HISTORY

    gr.Markdown("---")
    gr.Markdown("## 📚 Recognition History")

    history_output = gr.HTML(
        "<p>No recognition history available.</p>"
    )

    # DOWNLOAD REPORT

    gr.Markdown("---")
    gr.Markdown("## 📥 Download Recognition Report")

    report_output = gr.File(
        label="Download Bird Recognition Report"
    )

    # FOOTER

    gr.HTML("""
    <div style="
        background:#064e3b;
        color:white;
        padding:25px;
        text-align:center;
        border-radius:12px;
        margin-top:20px;">

        <h3 style="color:white">
        🐦 BIRDVISION AI
        </h3>

        Python | Gradio | Hugging Face | CLIP

        <br><br>
        Intelligent Bird Species Recognition System
    </div>
    """)

    # ANALYZE BUTTON

    analyze_btn.click(
        fn=recognize_bird,
        inputs=[
            image_input,
            history_state
        ],
        outputs=[
            result_output,
            chart_output,
            prediction_table,
            history_output,
            history_state,
            report_output
        ]
    )

    # RESET BUTTON

    reset_btn.click(
        fn=reset_dashboard,
        inputs=[],
        outputs=[
            image_input,
            result_output,
            chart_output,
            prediction_table,
            history_output,
            history_state,
            report_output
        ]
    )


# ============================================================
# 8. LAUNCH APPLICATION
# ============================================================

print("Launching BirdVision AI...")

demo.queue()

demo.launch(
    share=True,
    debug=False,
    show_error=True
)


# ============================================================
# PROJECT COMPLETED
# ============================================================


Loading BirdVision AI model...
Device: CPU


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

... (6 outputs left out)

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

AI MODEL LOADED SUCCESSFULLY!


/tmp/ipykernel_1209/419150586.py:459: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Launching BirdVision AI...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4e6e10cf1546e45fa4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
